# Predicted decoding accuracy: Fisher information + Monte Carlo simulate-and-decode

Two analyses of the same fitted nPRF + residual covariance, joined onto Figure 2 of the paper:

- **Fisher information** — local curvature of the log-likelihood at the true stimulus. Closed-form, fast, but assumes Gaussian-like noise behaviour.
- **Monte Carlo simulate-and-decode** — draw stimuli, simulate neural responses (with the fitted t-distributed residual), invert with the model's posterior, repeat 1000×. More robust when PRFs are wide or the residual is heavy-tailed.

The two should agree where Fisher's regularity conditions hold (narrow PRFs, low dof correction). Where they diverge, the MC version is the one to trust.

Computation:
- `tms_risk/modeling/fisher_information.py` → `derivatives/fisher_information.denoise/sub-*/ses-*/func/*.tsv`
- `tms_risk/modeling/monte_carlo_decode.py` → `derivatives/monte_carlo_decode.denoise/sub-*/ses-*/func/*.tsv`

SLURM submission: `tms_risk/modeling/slurm_jobs/submit_mc_decode.sh`.

In [ ]:
import os.path as op
from glob import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tms_risk.utils.data import get_tms_subjects, get_tms_conditions

bids_folder = '/data/ds-tmsrisk'
roi = 'NPC12r'  # mask used for the paper's main analyses
n_voxels = 100

In [ ]:
# Map session 2/3 → stimulation condition per subject
tms_keys = get_tms_conditions()
subjects = get_tms_subjects(bids_folder=bids_folder, exclude_outliers=True)
print(f'{len(subjects)} subjects (outliers 22, 49 dropped)')

## Load Fisher information

In [ ]:
def load_long(kind, glob_template):
    rows = []
    for sub in subjects:
        for session in (2, 3):
            path = glob_template.format(sub=f'{sub:02d}', ses=session,
                                          roi=roi, n=n_voxels)
            matches = glob(path)
            if not matches:
                continue
            df = pd.read_csv(matches[0], sep='\t', index_col=0)
            stim_cond = tms_keys[str(sub)][str(session)]
            df['subject'] = sub
            df['session'] = session
            df['stimulation_condition'] = stim_cond
            df['kind'] = kind
            rows.append(df)
    return pd.concat(rows, ignore_index=False) if rows else pd.DataFrame()


fisher = load_long(
    'fisher',
    op.join(bids_folder, 'derivatives', 'fisher_information.denoise',
            'sub-{sub}', 'ses-{ses}', 'func',
            'sub-{sub}_ses-{ses}_roi-{roi}_nvoxels-{n}_fisher_information.tsv')
)
mc = load_long(
    'mc',
    op.join(bids_folder, 'derivatives', 'monte_carlo_decode.denoise',
            'sub-{sub}', 'ses-{ses}', 'func',
            'sub-{sub}_ses-{ses}_roi-{roi}_nvoxels-{n}_mc_decode.tsv')
)
print(f'Fisher: {len(fisher):,} rows from {fisher["subject"].nunique() if len(fisher) else 0} subjects')
print(f'MC:     {len(mc):,} rows from {mc["subject"].nunique() if len(mc) else 0} subjects')

## Fisher information: precision vs magnitude × stimulation

In [ ]:
# Fisher info TSVs from fisher_information.py have stimulus on the index;
# the value column is the Fisher precision at each stimulus.
fisher_long = fisher.reset_index().rename(columns={'index': 'stimulus'})
fisher_long['stimulus'] = fisher_long['stimulus'].astype(float)
value_col = [c for c in fisher_long.columns if c.startswith('0') or c == 'fi']
value_col = value_col[0] if value_col else fisher_long.columns[0]
fisher_long = fisher_long.rename(columns={value_col: 'fisher'})

g = sns.relplot(
    data=fisher_long, x='stimulus', y='fisher',
    hue='stimulation_condition', kind='line',
    palette={'vertex': 'tab:green', 'ips': 'tab:red'},
    errorbar=('ci', 95), height=4, aspect=1.3,
)
g.set_axis_labels('Payoff magnitude', 'Fisher information (precision)')
g.set_titles('Predicted decoding precision (Fisher)')
plt.show()

## Monte Carlo: decoded SD and bias vs true magnitude

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)

sns.lineplot(
    data=mc, x='true_stim', y='decoded_sd',
    hue='stimulation_condition',
    palette={'vertex': 'tab:green', 'ips': 'tab:red'},
    errorbar=('ci', 95), ax=axes[0],
)
axes[0].set_ylabel('Decoded posterior SD')
axes[0].set_title('Decoding noise (MC, 1000 reps)')

sns.lineplot(
    data=mc, x='true_stim', y='bias',
    hue='stimulation_condition',
    palette={'vertex': 'tab:green', 'ips': 'tab:red'},
    errorbar=('ci', 95), ax=axes[1],
)
axes[1].axhline(0, ls='--', color='grey', lw=0.8)
axes[1].set_ylabel('Decoded − true (bias)')
axes[1].set_title('Decoding bias (MC, 1000 reps)')

for ax in axes:
    ax.set_xlabel('True payoff magnitude')

plt.tight_layout()
plt.show()

## Sanity check: Fisher ↔ MC agreement

If both procedures are well-specified, the MC-decoded SD should track $1/\sqrt{\text{Fisher}}$ across magnitudes (Cramér–Rao bound when achieved). Mismatch is informative — it usually means the empirical residual has tails Fisher's local quadratic underestimates.

In [ ]:
agg_fisher = (fisher_long.groupby(['stimulus', 'stimulation_condition'])
                ['fisher'].mean().reset_index())
agg_fisher['cramer_rao_sd'] = 1.0 / np.sqrt(agg_fisher['fisher'])

agg_mc = (mc.groupby(['true_stim', 'stimulation_condition'])
            ['decoded_sd'].mean().reset_index()
            .rename(columns={'true_stim': 'stimulus'}))

compare = agg_fisher.merge(agg_mc, on=['stimulus', 'stimulation_condition'])

fig, ax = plt.subplots(figsize=(5.5, 5))
for cond, color in [('vertex', 'tab:green'), ('ips', 'tab:red')]:
    sub = compare[compare['stimulation_condition'] == cond]
    ax.scatter(sub['cramer_rao_sd'], sub['decoded_sd'],
                label=cond, color=color, alpha=0.7)
lo = min(compare['cramer_rao_sd'].min(), compare['decoded_sd'].min())
hi = max(compare['cramer_rao_sd'].max(), compare['decoded_sd'].max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, label='identity')
ax.set_xlabel('1 / √(Fisher info)  [Cramér–Rao SD]')
ax.set_ylabel('Monte Carlo decoded SD')
ax.set_title('Fisher ↔ MC agreement (Cramér–Rao bound)')
ax.legend()
plt.show()